In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("01 covid_toy - 01 covid_toy.csv")

display(df.head())

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [3]:
df = df[["age", "gender", "fever", "has_covid"]]

df = df.drop_duplicates()

print(df.head())
print(df.isnull().sum())

   age  gender  fever has_covid
0   60    Male  103.0        No
1   27    Male  100.0       Yes
2   42    Male  101.0        No
3   31  Female   98.0        No
4   65  Female  101.0        No
age           0
gender        0
fever        10
has_covid     0
dtype: int64


In [5]:
X = df[["age", "gender", "fever"]]
y = df["has_covid"]

numeric_features = ["age", "fever"]
categorical_features = ["gender"]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

In [6]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.5263157894736842
              precision    recall  f1-score   support

          No       0.54      0.70      0.61        10
         Yes       0.50      0.33      0.40         9

    accuracy                           0.53        19
   macro avg       0.52      0.52      0.50        19
weighted avg       0.52      0.53      0.51        19



In [7]:
new_person = pd.DataFrame({
    "age": [25],
    "gender": ["Male"],
    "fever": [101.5]
})

prediction = model.predict(new_person)

print("COVID Prediction:", prediction[0])

COVID Prediction: No
